# Create a semantic document vector database for RAG

## What is a vector database?

A vector database, like ChromaDB, stores lists of vectors (arrays of real numbers) and associated information.

A vector database is used to quickly query for similar vectors. 

## How can you generate a vector representing the semantics of a chunk of text?

What is a given piece of text about? Embeddings models have answers, for example: [1.25, -0.22, 3.55, ...]

Okay, that seems a little odd, but one can imagine that the embeddings model are converting a piece of text to a vector pointing at some point in an "embeddings space". The embeddings space represents many semantic categories as directions in embeddings space (this can be thought of as axes, but in fact there are exponentially more nearly-orthogonal directions than there are axes in the embeddings space, which the model leverages).

Thus, chunks of text about similar topics will typically be closer in embeddings space than unrelated chunks. 

## Where this is going: Retrieval-Augmented Generation

Similarly, we can take users' LLM queries and run them through the same embeddings model, and generate another vector. By finding similar document vectors to the query vector, we obtain the most relevant documents.

## Implementation Details

We're dealing with a handful of large Markdown documents containing a variety of information about the CHPC Student Cluster Competition selection round.

### Document splitting

We don't want to place a large amount of irrelevant documents into model context. This wastes the limited context window and makes it more likely that the LLM may miss key facts in the haystack of retrieved text. 

The source documents cover many different topics, so its best to break them up into smaller chunks so that the RAG process only retrieves the most relevant chunks.

In [1]:
import os
import subprocess
import shutil
from datetime import datetime

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
import chromadb

from retrieval_split_text import split_source_text_in_dir

embeddings_model_name = "sentence-transformers/all-distilroberta-v1"

RAW_DIR = "/opt/shared/data/raw"
DOC_DIR = "/opt/shared/data/split"
VDB_DIR = "/opt/shared/data/chromadb"

In [4]:
shutil.rmtree(RAW_DIR, ignore_errors=True)

print("Syncing raw document data...")

subprocess.run("../pull_data.sh")

print("Splitting documents...")

shutil.rmtree(DOC_DIR, ignore_errors=True)
split_source_text_in_dir(RAW_DIR, DOC_DIR)

print(f"Loading the sentence transformer, {embeddings_model_name}...")

embeddings_model = SentenceTransformer(embeddings_model_name)

print("Loading ChromaDB and the documents...")

chroma = chromadb.PersistentClient(path=VDB_DIR)

try:
    chroma.delete_collection(name="all-documents")
except:
    pass

vector_collection = chroma.create_collection(
    name="all-documents",
    # We generate the embeddings using sentence-transformers directly for pedagogical reasons
    embedding_function=None,
    configuration={
        "hnsw": {
            # Cosine similarity is most useful for text embeddings I believe,
            # where scale is of little importance?
            "space": "cosine",
        }
    }
)

print("Generating embeddings and storing documents...")

paths = []
docs = []
for entry in os.scandir(DOC_DIR):
    if entry.is_file():
        paths.append(entry.path)

        with open(entry.path, "r") as f:
            docs.append(f.read())

embeddings = embeddings_model.encode(docs)

fnames = [os.path.basename(path) for path in paths]

vector_collection.add(
    ids=fnames,
    embeddings=embeddings,
    documents=docs,
)

print("Done!")

Syncing raw document data...
Already up to date.
Extracting the markdown files
Copied: /opt/shared/data/_selection_round_github/tutorial1/README.md -> /opt/shared/data/raw/tutorial1_README.md
Copied: /opt/shared/data/_selection_round_github/tutorial4/README.md -> /opt/shared/data/raw/tutorial4_README.md
Copied: /opt/shared/data/_selection_round_github/tutorial2/README.md -> /opt/shared/data/raw/tutorial2_README.md
Copied: /opt/shared/data/_selection_round_github/README.md -> /opt/shared/data/raw/README.md
Copied: /opt/shared/data/_selection_round_github/tutorial3/README.md -> /opt/shared/data/raw/tutorial3_README.md
Splitting documents...
splitting tutorial4_README.md
splitting tutorial3_README.md
splitting tutorial2_README.md
splitting README.md
splitting tutorial1_README.md
Loading the sentence transformer, sentence-transformers/all-distilroberta-v1...
Loading ChromaDB and the documents...
Generating embeddings and storing documents...
Done!
